In [ ]:
#@title Preparar entorno e interactividad { display-mode: "form" }
import json, html as html_lib, base64
from IPython.display import display, HTML

def tutorial(url, alto=760, titulo="Tutorial"):
    sep = "&" if "?" in url else "?"
    src = f"{url}{sep}embed=1"
    display(HTML(f'''
    <iframe src="{src}" width="100%" height="{alto}"
            style="border:0;display:block;border-radius:10px;background:#faf7f0;"
            loading="lazy" title="{html_lib.escape(titulo)}"></iframe>
    <p style="margin:8px 0 0;">
      <a href="{url}" target="_blank" rel="noopener">
        Abrir {html_lib.escape(titulo)} en pantalla completa ↗
      </a>
    </p>
    '''))

def pregunta_interactiva(numero, tema, pregunta, opciones, correcta, retro):
    uid = f"s05-p{numero}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0;"><input type="radio" name="{uid}" value="{i}"> '
        f'{html_lib.escape(op)}</label>'
        for i, op in enumerate(opciones)
    )
    retro_json = json.dumps(retro, ensure_ascii=False)
    display(HTML(f'''
    <div style="border:2px solid #175c3c;border-radius:12px;padding:16px;margin:14px 0;background:#f4faf6;color:#172019;">
      <div style="font-weight:700;color:#123f2b;margin-bottom:8px;">Pregunta {numero} · {html_lib.escape(tema)}</div>
      <p><strong>{html_lib.escape(pregunta)}</strong></p>
      {opts}
      <button onclick="(function(){{
          const e=document.querySelector('input[name={uid}]:checked');
          const s=document.getElementById('r-{uid}');
          if(!e){{s.innerHTML='Selecciona una opción.';return;}}
          const i=Number(e.value); const r={retro_json};
          const ok=i==={correcta};
          s.innerHTML='<div style=&quot;margin-top:10px;padding:10px;border-radius:8px;background:'+
            (ok?'#d1e7dd;color:#0f5132':'#f8d7da;color:#842029')+'&quot;><strong>'+
            (ok?'Correcto. ':'Revisa. ')+'</strong>'+r[i]+'</div>';
        }})()"
        style="background:#175c3c;color:white;border:0;border-radius:7px;padding:8px 13px;cursor:pointer;">
        Verificar
      </button>
      <div id="r-{uid}" aria-live="polite"></div>
    </div>
    '''))
print("Entorno de la sesión 5 listo.")


def pregunta_interactiva_codificada(payload_b64):
    payload = json.loads(base64.b64decode(payload_b64).decode("utf-8"))
    pregunta_interactiva(
        payload["numero"], payload["tema"], payload["pregunta"],
        payload["opciones"], payload["correcta"], payload["retro"]
    )

<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/5_Atlas_Cassandra_Query_First.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir la sesión 5 en Google Colab">
</a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/)

> **Punto de partida real.** La sesión anterior terminó después de crear `compras_claras` en Atlas
> y cargar las colecciones `noticias` y `entidades_noticias`. Hoy continuamos desde ahí:
> no repetimos el registro ni la carga.

# Sesión 5 — De una vista de Atlas a una consulta operacional con Cassandra

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Fecha:** 3 de septiembre de 2026  
**Tema ajustado del PDA:** práctica de Cassandra, preservando el cierre pendiente de MongoDB Atlas  
**Caso conductor:** Compras Claras  
**Pregunta profesional:** **¿cómo convierte Laura una priorización analítica en una consulta que pueda repetir sin reconstruir todo cada vez?**

### Producto observable de hoy

Al terminar tendrás:

1. una vista `menciones_clasificadas` construida en Atlas;
2. la evidencia `111 / 25 / 6` por nivel de mención;
3. la regla reproducible `1.000 → 163 → 77`;
4. el límite analítico `0 de 77` referencias de proceso citadas literalmente en prensa;
5. una tabla Cassandra diseñada **desde una consulta**;
6. CRUD ejecutado con CQL y, si tu conexión está lista, también desde Python;
7. una consulta que el modelo responde bien y otra que **no** responde bien;
8. un hito descargable con tu decisión y su límite;
9. `s05_ancla_s06.json`: el proceso que Laura abrirá en la sesión 6 para estudiar su contexto relacional.

### CONTINUIDAD S03-S05 — del prototipo a la bandeja operacional

En S3 apareció una primera bandeja de **200 procesos**. Ese resultado fue un **prototipo exploratorio**: servía para demostrar que las señales podían reducir el universo, pero todavía mezclaba decisiones que no habíamos convertido en una regla operacional estable.

Hoy no estamos “corrigiendo 200 por 77” ni comparando el mismo indicador. En S5 fijamos una regla distinta, explícita y reproducible sobre la muestra de 1.000 procesos:

```text
entidad presente en prensa
+ modalidad contiene "directa"
+ respuestas al procedimiento = 0
────────────────────────────────
77 candidatos
```

**PARA LLEVAR.** El número importante no es 77 por sí solo. Lo importante es que Laura puede explicar exactamente **cómo entró cada proceso** y qué evidencia todavía falta.

## Mapa de la sesión

| Bloque | Pregunta | Qué queda |
|---|---|---|
| 1. Retomar Atlas | ¿qué falta entre dos colecciones y una decisión? | pipeline y vista |
| 2. Tutorial visual Atlas | ¿dónde se construye y guarda? | `menciones_clasificadas` |
| 3. Volver a Colab | ¿qué procesos pasan primero? | 1.000 → 163 → 77 |
| 4. Límite | ¿qué evidencia tenemos realmente? | 0/77 |
| 5. Cassandra | ¿cómo servir una pregunta repetitiva? | query-first design |
| 6. Tutorial visual Astra | ¿dónde creo y pruebo el modelo? | tabla + CQL |
| 7. Python | ¿cómo lo usa una aplicación? | CRUD con driver |
| 8. Cierre | ¿qué resolvió cada motor? | decisión defendible |

**Regla de navegación del curso:** el cuaderno explica **por qué, qué significa y qué límite tiene**.
Las presentaciones HTML embebidas explican **dónde hacer clic y qué debe aparecer**.

---
## 1. Reactivar sin repetir la sesión 4

Antes de entrar a consultas, verifica el estado esperado:

```text
Atlas
└── compras_claras
    ├── noticias               → 987 documentos
    └── entidades_noticias     → 142 documentos
```

La siguiente celda solo recupera la conexión. La contraseña se pide con `getpass()` y no queda escrita.

<details>
<summary><strong>Si no conservas la URI de S4</strong></summary>

1. En Atlas abre tu clúster → **Connect / Drivers**.
2. Selecciona **Python** y copia la URI con `<db_username>` y `<db_password>`.
3. Vuelve aquí. **No crees otra base ni otro clúster.**

Si tampoco recuerdas el camino, usa `assets/tutoriales/atlas-guia-conexion.html` solo como recuperación.
</details>

In [ ]:
!pip install -q pymongo dnspython

from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

uri_pegada = input("Pega tu URI de Atlas (Connect / Drivers): ").strip()
if not uri_pegada:
    raise ValueError("La URI está vacía. Recupera la URI de tu clúster de S4.")

uri = uri_pegada
if "<db_username>" in uri or "<db_password>" in uri:
    usuario = input("Usuario de base de datos: ").strip()
    contrasena = quote_plus(getpass("Contraseña (no se muestra): "))
    uri = uri.replace("<db_username>", quote_plus(usuario))
    uri = uri.replace("<db_password>", contrasena)

try:
    client = MongoClient(uri, serverSelectionTimeoutMS=7000)
    client.admin.command("ping")
    db = client["compras_claras"]
    motor_atlas = "Atlas real"
    print("Conectado.")
    print("noticias:", db["noticias"].count_documents({}))
    print("entidades_noticias:", db["entidades_noticias"].count_documents({}))
except Exception as error:
    db = None
    motor_atlas = "respaldo por archivos"
    print("No se pudo conectar:", type(error).__name__)
    print("La clase puede continuar con respaldo; eso NO sustituye haber creado la vista en Atlas.")

print("Modo:", motor_atlas)

## 2. De documentos a una vista que sí usaremos

| Objeto | Pregunta mental | ¿Guarda datos nuevos? |
|---|---|---|
| filtro / `find()` | ¿qué documentos quiero ver? | no |
| pipeline | ¿qué transformaciones quiero encadenar? | no |
| pipeline guardado | ¿quiero conservar la receta? | no |
| vista | ¿quiero consultar el resultado como objeto de solo lectura? | no |

La historia necesita una transformación: convertir menciones por entidad en una señal explicable.

| Etapa | Para qué sirve aquí | Qué debes mirar |
|---|---|---|
| `$set` | agrega `nivel_menciones` | crea un campo |
| `$switch` | aplica los cortes 20 y 5 | primera condición verdadera gana |
| `$project` | deja los campos útiles | reduce ruido |
| `$sort` | ordena la salida | la hace legible |

| `noticias` | nivel esperado |
|---:|---|
| 4 | baja |
| 5 | media |
| 19 | media |
| 20 | alta |

**PARA LLEVAR.** El pipeline es la receta; la vista publica esa receta como salida consultable de solo lectura.

In [ ]:
#@title Pregunta 1 — Pipeline y vista { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjEsInRlbWEiOiJQaXBlbGluZSB5IHZpc3RhIiwicHJlZ3VudGEiOiLCv0N1w6FsIGFmaXJtYWNpw7NuIGRlc2NyaWJlIG1lam9yIHVuYSB2aXN0YSBjcmVhZGEgZGVzZGUgdW4gcGlwZWxpbmUgZGUgYWdyZWdhY2nDs24/Iiwib3BjaW9uZXMiOlsiRHVwbGljYSBsb3MgZG9jdW1lbnRvcyBwYXJhIHF1ZSBsYSBjb25zdWx0YSBzZWEgbcOhcyByw6FwaWRhLiIsIkNvbnNlcnZhIHVuYSBkZWZpbmljacOzbiBkZSBzb2xvIGxlY3R1cmEgcXVlIHNlIGV2YWzDumEgZGVzZGUgbG9zIGRhdG9zIGRlIG9yaWdlbi4iLCJFcyBleGFjdGFtZW50ZSBsbyBtaXNtbyBxdWUgZ3VhcmRhciB1biBwaXBlbGluZSBlbiBlbCBlZGl0b3IuIiwiUGVybWl0ZSBlc2NyaWJpciBzb2JyZSBlbCByZXN1bHRhZG8gc2luIG1vZGlmaWNhciBsYSBjb2xlY2Npw7NuIGRlIG9yaWdlbi4iXSwiY29ycmVjdGEiOjEsInJldHJvIjpbIk5vLiBVbmEgdmlzdGEgZXN0w6FuZGFyIG5vIGNyZWEgdW5hIGNvcGlhIGluZGVwZW5kaWVudGUgZGUgbG9zIGRvY3VtZW50b3MuIiwiU8OtLiBMYSB2aXN0YSBjb25zZXJ2YSBsYSBkZWZpbmljacOzbiB5IGV4cG9uZSBzdSByZXN1bHRhZG8gY29tbyB1biBvYmpldG8gY29uc3VsdGFibGUgZGUgc29sbyBsZWN0dXJhLiIsIk5vLiBHdWFyZGFyIGxhIHJlY2V0YSB5IGNyZWFyIHVuYSB2aXN0YSBzb24gb3BlcmFjaW9uZXMgZGlzdGludGFzLiIsIk5vLiBMYSB2aXN0YSBlc3TDoW5kYXIgZXMgZGUgc29sbyBsZWN0dXJhLiJdfQ==")

---
## 3. Tutorial visual 1 — Atlas: construir la vista que sí viaja

**HAZ ESTO AHORA.** Trabaja en Atlas y vuelve a este mismo cuaderno.

No repetimos registro, clúster, carga, filtros de calentamiento ni pipelines que no se consumen después.

Al regresar deben existir:

- pipeline guardado `clasificar-menciones-v1`;
- vista **`menciones_clasificadas`**;
- control `6 alta + 25 media + 111 baja = 142`.

**MÁS ADELANTE.** `resumen-secciones-v1` y `clasificar-noticias-v1` quedan como ampliación para recuperar tiempo de laboratorio.

In [ ]:
#@title Tutorial 1 — Atlas: vista paso a paso { display-mode: "form" }
tutorial(
    "https://jazaineam1.github.io/BigData2026/assets/tutoriales/atlas-s05-pipelines-vistas-v2.html",
    alto=780,
    titulo="Tutorial Atlas — consultas, pipelines y vistas"
)

### Control antes de avanzar

No continúes por intuición. Debes poder responder **sí** a estas tres preguntas:

- [ ] ¿veo `menciones_clasificadas` como una vista?
- [ ] ¿puedo explicar qué hace `$switch` en el pipeline?
- [ ] ¿sé la diferencia entre guardar el pipeline y crear la vista?

Si no, vuelve a la diapositiva correspondiente. El bloque siguiente consume esa vista.

---
## 4. Consumir la vista y comprobar su resultado

Ahora el cuaderno vuelve a ser el lugar donde **se integra** la evidencia.

Si la vista existe en Atlas, la traemos tal como la construiste.  
Si no existe o la conexión falló, se reconstruye el mismo resultado desde el archivo versionado para que nadie pierda el resto de la clase.

**El respaldo permite aprender; no prueba que hayas completado el paso de Atlas.**

In [ ]:
import json, urllib.request
from collections import Counter

vista_real = False

if db is not None:
    nombres = db.list_collection_names()
    if "menciones_clasificadas" in nombres:
        menciones = list(db["menciones_clasificadas"].find({}, {"_id": 0}))
        vista_real = True
    else:
        print("La vista no aparece en Atlas. Activo respaldo con la MISMA regla.")

if not vista_real:
    with urllib.request.urlopen("https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/entidades_en_noticias_2026.json") as r:
        menciones = json.loads(r.read().decode("utf-8"))
    for e in menciones:
        n = e.get("noticias", 0)
        e["nivel_menciones"] = "alta" if n >= 20 else "media" if n >= 5 else "baja"

niveles = Counter(m["nivel_menciones"] for m in menciones)
print("Fuente:", "vista real de Atlas" if vista_real else "respaldo reproducible")
print("Entidades:", len(menciones))
print("Niveles:", dict(niveles))

assert len(menciones) == 142
assert dict(niveles) == {"baja": 111, "media": 25, "alta": 6}

### Cómo leer esta salida

**Cómo se lee.** De 142 entidades asociadas a las noticias, 6 quedan en nivel alto, 25 en medio y 111 en bajo según una regla basada en número de noticias.

**Qué nos dice.** Ya tenemos una variable explicable que resume intensidad de mención **por entidad**.

**Qué NO permite concluir todavía.** No sabemos si una noticia habla de un contrato específico. Nos falta comparar referencias de proceso.

**Error frecuente.** Leer “nivel alto” como “riesgo alto”. Aquí solo significa **más menciones según el umbral que definimos**.

---
## 5. Atlas + SECOP: convertir una vista en una bandeja

La vista por sí sola no responde la pregunta de Laura. La cruzamos con una muestra de 1.000 procesos SECOP mediante una regla explícita:

```text
1. entidad aparece en prensa
2. modalidad contiene "directa"
3. respuestas al procedimiento = 0
4. ordenar por precio_base DESC
```

No es un modelo predictivo. Es una regla transparente que cualquiera puede discutir.

In [ ]:
import pandas as pd

secop = pd.read_csv(
    "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
    low_memory=False,
)
print("Procesos SECOP:", len(secop))
assert len(secop) == 1000

contexto_menciones = pd.DataFrame(menciones)[["entidad", "noticias", "nivel_menciones"]].copy()
assert contexto_menciones["entidad"].is_unique
contexto_menciones = contexto_menciones.rename(columns={"noticias": "noticias_entidad"})

entidades_en_prensa = set(contexto_menciones["entidad"])
paso1 = secop[secop["entidad"].isin(entidades_en_prensa)]
paso2 = paso1[
    paso1["modalidad_de_contratacion"].str.contains("directa", case=False, na=False)
]
respuestas = pd.to_numeric(paso2["respuestas_al_procedimiento"], errors="coerce")
paso3 = paso2[respuestas.eq(0)]

candidatos = (
    paso3
    .merge(contexto_menciones, on="entidad", how="left", validate="many_to_one")
    .sort_values(["precio_base", "id_del_proceso"], ascending=[False, True])
    .reset_index(drop=True)
)

print("1) total                       :", len(secop))
print("2) entidad coincide con prensa :", len(paso1))
print("3) + modalidad directa         :", len(paso2))
print("4) + cero respuestas           :", len(paso3))
print("Candidatos finales             :", len(candidatos))
print("Contexto de menciones en los 77:", candidatos["nivel_menciones"].value_counts().to_dict())

assert len(paso1) == 163
assert len(candidatos) == 77
assert candidatos["nivel_menciones"].notna().all()

### INTERPRETACIÓN EMBUDO S05

**Cómo se lee.** Partimos de 1.000 procesos, 163 pertenecen a entidades presentes en noticias y 77 sobreviven a modalidad directa + cero respuestas. Cada candidato conserva `noticias_entidad` y `nivel_menciones` de la vista.

**Qué nos dice.** La vista aporta contexto explicable a cada fila que Laura recibirá.

**Qué NO permite concluir todavía.** `nivel_menciones` no es criterio de selección ni probabilidad de riesgo; todavía no sabemos si una noticia menciona el contrato particular.

**Error frecuente.** Creer que “alta” empujó un proceso dentro de los 77. El nivel viaja como contexto, no como filtro.

### El detalle estadístico que importa: faltante no es cero

La celda anterior usa:

```python
pd.to_numeric(..., errors="coerce").eq(0)
```

y **no** usa:

```python
fillna(0).eq(0)
```

¿Por qué? Porque “no conocemos el número de respuestas” y “hubo exactamente cero respuestas” son eventos distintos.
Convertir el faltante a cero fabricaría evidencia que no existe.

In [ ]:
primero = candidatos.iloc[0]

print("Primer candidato")
print("ID                :", primero["id_del_proceso"])
print("Entidad           :", primero["entidad"])
print("Valor             : $", f'{primero["precio_base"]:,.0f}')
print("Noticias entidad  :", int(primero["noticias_entidad"]))
print("Nivel de menciones:", primero["nivel_menciones"])

assert primero["id_del_proceso"] == "CO1.REQ.5407319"
assert primero["entidad"] == "MINISTERIO DEL DEPORTE"
assert int(primero["precio_base"]) == 168750000

---
## 6. El límite que debe viajar con la bandeja

Ahora hacemos una prueba incómoda: buscar si la **referencia exacta del proceso** aparece literalmente en el título o subtítulo de alguna noticia.

Si la respuesta fuera alta, tendríamos evidencia a nivel de contrato.  
Si es cero, la evidencia sigue siendo a nivel de **entidad**.

In [ ]:
with urllib.request.urlopen("https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_contratacion_2026.json") as r:
    noticias = json.loads(r.read().decode("utf-8"))

texto_prensa = " ".join(
    f'{n.get("titulo") or ""} {n.get("subtitulo") or ""}'
    for n in noticias
).casefold()

def citado_literalmente(fila):
    referencia = str(fila.get("referencia_del_proceso") or "").strip()
    return bool(referencia) and len(referencia) >= 6 and referencia.casefold() in texto_prensa

candidatos = candidatos.copy()
candidatos["referencia_citada_en_prensa"] = candidatos.apply(citado_literalmente, axis=1)

con_referencia = int(candidatos["referencia_citada_en_prensa"].sum())
print("Referencias citadas literalmente:", con_referencia, "de", len(candidatos))

assert con_referencia == 0

### Cómo leer esta salida

**Cómo se lee.** Ninguno de los 77 candidatos tiene su referencia de proceso citada literalmente en los títulos o subtítulos de las noticias usadas.

**Qué nos dice.** La regla puede ayudar a decidir **dónde mirar primero**.

**Qué NO permite concluir todavía.** No demuestra que esos 77 procesos hayan sido cuestionados por la prensa, ni fraude, irregularidad o incumplimiento.

**Error frecuente.** Decir “la prensa señaló estos contratos”. No: la evidencia periodística que usamos está asociada a la **entidad**, no al contrato específico.

In [ ]:
#@title Pregunta 2 — Qué podemos afirmar { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjIsInRlbWEiOiJMw61taXRlIGFuYWzDrXRpY28iLCJwcmVndW50YSI6IsK/Q3XDoWwgYWZpcm1hY2nDs24gcHVlZGUgc29zdGVuZXIgTGF1cmEgZGVzcHXDqXMgZGVsIDAgZGUgNzc/Iiwib3BjaW9uZXMiOlsiRW5jb250cmFtb3MgNzcgY29udHJhdG9zIGlycmVndWxhcmVzLiIsIkxhIHByZW5zYSBpZGVudGlmaWPDsyBkaXJlY3RhbWVudGUgbG9zIDc3IGNvbnRyYXRvcy4iLCJQcmlvcml6YW1vcyA3NyBwcm9jZXNvcyBwb3Igc2XDsWFsZXMgZXhwbGljYWJsZXM7IGxhIGV2aWRlbmNpYSBkZSBwcmVuc2EgZXMgcG9yIGVudGlkYWQgeSByZXF1aWVyZSByZXZpc2nDs24gaHVtYW5hLiIsIkNvbW8gMCBkZSA3NyBlc3TDoSBjaXRhZG8sIGxhIHJlZ2xhIG5vIHNpcnZlIHBhcmEgbmFkYS4iXSwiY29ycmVjdGEiOjIsInJldHJvIjpbIk5vLiBVbmEgcmVnbGEgZGUgcHJpb3JpZGFkIG5vIHBydWViYSBpcnJlZ3VsYXJpZGFkLiIsIk5vLiBFbCBjb250cm9sIDAvNzcgbXVlc3RyYSBqdXN0YW1lbnRlIGxvIGNvbnRyYXJpby4iLCJDb3JyZWN0by4gTGEgc2FsaWRhIHNpcnZlIHBhcmEgcHJpb3JpemFyLCB5IHN1IGzDrW1pdGUgdmlhamEgY29uIGVsbGEuIiwiTm8uIEVsIGzDrW1pdGUgcmVkdWNlIGxhIGZ1ZXJ6YSBkZSBsYSBhZmlybWFjacOzbiwgbm8gZWxpbWluYSBlbCB2YWxvciBvcGVyYXRpdm8gZGUgcHJpb3JpemFyLiJdfQ==")

---
# 7. Cassandra aparece por una necesidad, no por el cronograma

Laura ya tiene una bandeja. Imagina ahora este patrón:

> “Para **este corte** y **este departamento**, dame los **5 procesos de mayor valor** que debo revisar primero.”

Hoy pregunta una persona. Mañana pueden ser cien analistas consultando lo mismo una y otra vez.

Con 77 filas **no necesitas Cassandra**: pandas responde sobrado.  
La razón de estudiarlo es aprender un patrón que sigue funcionando cuando la bandeja crece y la misma consulta se repite a escala.

## Query-first design

En Cassandra no empezamos preguntando “¿qué entidades existen?”. Empezamos por:

> **¿Qué consulta debo servir de forma barata y repetitiva?**

Luego diseñamos la tabla alrededor de ella.

### EJERCICIO S05-PK — Elige antes de mirar la respuesta

La consulta profesional es:

> **Para un corte y un departamento, devolver primero los procesos de mayor valor.**

¿Cuál diseño permite localizar directamente el grupo que Laura conoce al consultar?

- **A.** `PRIMARY KEY (id_proceso)`
- **B.** `PRIMARY KEY ((corte, departamento), valor_base, id_proceso)`
- **C.** `PRIMARY KEY ((entidad), id_proceso)`

No es una pregunta calificable. La decisión importa más que acertar de memoria: elige una opción y explica en una frase por qué descartas otra.

In [ ]:
eleccion_pk = input("Tu elección (A, B o C): ").strip().upper()
razon_descartada = input("Descarta una alternativa en una frase: ").strip()

if eleccion_pk == "B":
    print("Bien: la consulta conoce corte + departamento y puede localizar esa partición.")
elif eleccion_pk in {"A", "C"}:
    print("Revisa el patrón de acceso: Laura conoce corte + departamento, no un id ni necesariamente una entidad.")
else:
    print("Escribe A, B o C. Luego vuelve a leer la pregunta profesional.")

print("Alternativa descartada:", razon_descartada or "PENDIENTE")

### La llave se lee en dos partes

```sql
PRIMARY KEY (
    (corte, departamento),
    valor_base,
    id_proceso
)
```

| Parte | Papel | Pregunta |
|---|---|---|
| `(corte, departamento)` | clave de partición | ¿a qué partición debo ir? |
| `valor_base` | clustering | ¿cómo quedan ordenadas las filas dentro de la partición? |
| `id_proceso` | clustering/desempate | ¿cómo identifico una fila de forma estable si dos valores empatan? |

Con:

```sql
CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC)
```

la partición ya queda preparada para responder:

```sql
WHERE corte = ? AND departamento = ?
LIMIT 5
```

sin tener que ordenar todo después.

In [ ]:
#@title Pregunta 3 — Clave de partición { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjMsInRlbWEiOiJRdWVyeS1maXJzdCBkZXNpZ24iLCJwcmVndW50YSI6IsK/UG9yIHF1w6kgbGEgdGFibGEgdXNhIChjb3J0ZSwgZGVwYXJ0YW1lbnRvKSBjb21vIGNsYXZlIGRlIHBhcnRpY2nDs24/Iiwib3BjaW9uZXMiOlsiUG9ycXVlIHNvbiBsYXMgY29sdW1uYXMgbcOhcyBpbXBvcnRhbnRlcyBkZWwgbmVnb2Npby4iLCJQb3JxdWUgbGEgY29uc3VsdGEgcmVwZXRpdGl2YSBjb25vY2UgZXNvcyBkb3MgdmFsb3JlcyB5IHB1ZWRlIGxvY2FsaXphciB1bmEgcGFydGljacOzbiBjb25jcmV0YS4iLCJQb3JxdWUgQ2Fzc2FuZHJhIGV4aWdlIGV4YWN0YW1lbnRlIGRvcyBjb2x1bW5hcyBlbiB0b2RhIGNsYXZlIGRlIHBhcnRpY2nDs24uIiwiUG9ycXVlIGFzw60gc2UgcHVlZGVuIGhhY2VyIGpvaW5zIGNvbiBNb25nb0RCLiJdLCJjb3JyZWN0YSI6MSwicmV0cm8iOlsiTm8gYmFzdGEgY29uIHF1ZSB1biBjYW1wbyBzZWEgaW1wb3J0YW50ZTogbGEgY2xhdmUgbmFjZSBkZWwgcGF0csOzbiBkZSBhY2Nlc28uIiwiQ29ycmVjdG8uIExhIHBhcnRpY2nDs24gc2UgZGlzZcOxYSBkZXNkZSBsYSBjb25zdWx0YSBxdWUgcXVlcmVtb3Mgc2VydmlyLiIsIk5vLiBVbmEgY2xhdmUgZGUgcGFydGljacOzbiBwdWVkZSB0ZW5lciB1bmEgbyB2YXJpYXMgY29sdW1uYXMuIiwiTm8uIENhc3NhbmRyYSBubyBpbnRyb2R1Y2Ugam9pbnMgY29uIE1vbmdvREI7IGxhIGJhbmRlamEgc2UgcHJlcGFyYSBhbnRlcy4iXX0=")

## ¿Para qué datos encaja este patrón?

| Encaja bien | No es la primera opción |
|---|---|
| pocas consultas conocidas de antemano | exploración donde la pregunta cambia cada cinco minutos |
| muchas escrituras y lecturas repetitivas | joins frecuentes entre entidades normalizadas |
| distribución horizontal y alta disponibilidad | transacciones relacionales complejas |
| datos desnormalizados para servir una pregunta | “guardar una vez y preguntar cualquier cosa después” |

**PARA LLEVAR.** Cassandra no es “MongoDB pero más rápido”. Hace otro compromiso:
te obliga a preparar el almacenamiento para unas consultas concretas.

### CHULETA CQL S05 — cinco comandos, una sola historia

| Comando | Para qué sirve hoy | Qué debes observar | Error frecuente |
|---|---|---|---|
| `USE compras_claras;` | trabajar en el keyspace creado en Astra | la consola cambia de contexto | intentar `CREATE KEYSPACE` en Astra |
| `CREATE TABLE` | definir la tabla para la consulta objetivo | la `PRIMARY KEY` nace del patrón de acceso | diseñar por columnas “importantes” |
| `INSERT INTO` | escribir una fila | columnas y valores corresponden | pensar que es una carga analítica masiva |
| `SELECT ... WHERE` | leer una partición | `WHERE` conoce `corte + departamento` | filtrar cualquier columna porque existe |
| `UPDATE` / `DELETE` | cambiar o borrar una fila identificada | se usa la clave necesaria para identificarla | olvidar parte de la clave |

**PARA LLEVAR.** CQL se parece visualmente a SQL; Cassandra no hereda por eso el mismo modelo de consultas ad hoc.

### CONTRATO DE RESULTADO S05 — fija primero qué debería devolver Cassandra

Elige un departamento mediante número. pandas calcula el top 5 esperado; después Cassandra debe devolver los mismos IDs y en el mismo orden.

In [ ]:
conteo_departamentos = candidatos["departamento_entidad"].dropna().astype(str).value_counts()
opciones_departamento = conteo_departamentos.index.tolist()

print("Elige un departamento para tu evidencia individual:")
for i, d in enumerate(opciones_departamento, start=1):
    print(f"{i:>2}. {d} ({conteo_departamentos[d]} candidatos)")

seleccion = int(input("Número de departamento: ").strip())
if not 1 <= seleccion <= len(opciones_departamento):
    raise ValueError("El número no corresponde a la lista mostrada.")

departamento_elegido = opciones_departamento[seleccion - 1]
top5_esperado_pd = (
    candidatos[candidatos["departamento_entidad"].astype(str) == departamento_elegido]
    .sort_values(["precio_base", "id_del_proceso"], ascending=[False, True])
    .head(5)
)
ids_esperados_pd = top5_esperado_pd["id_del_proceso"].astype(str).tolist()

print("Departamento elegido:", departamento_elegido)
print("IDs esperados por pandas:", ids_esperados_pd)

### RECUPERACIÓN S05 — si Colab reinició durante el receso

La siguiente celda es segura de ejecutar siempre. Si `candidatos` sigue en memoria, no hace nada.
Si el runtime se perdió, reconstruye la bandeja desde los archivos versionados y vuelve a comprobar
`142`, `111/25/6`, `1.000 → 163 → 77` y `0/77`.

**OJO.** Este respaldo recupera el trabajo analítico; no reemplaza la evidencia de haber creado la vista real en Atlas.

In [ ]:
#@title Recuperar la bandeja si Colab reinició { display-mode: "form" }
# RECUPERACIÓN S05
if "candidatos" not in globals():
    import json, urllib.request
    import pandas as pd
    from collections import Counter

    RAW = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main"

    with urllib.request.urlopen(f"{RAW}/Datos/entidades_en_noticias_2026.json") as r:
        menciones = json.loads(r.read().decode("utf-8"))
    for e in menciones:
        n = e.get("noticias", 0)
        e["nivel_menciones"] = "alta" if n >= 20 else "media" if n >= 5 else "baja"
    niveles = Counter(m["nivel_menciones"] for m in menciones)
    vista_real = False

    secop = pd.read_csv(
        f"{RAW}/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
        low_memory=False,
    )
    entidades_en_prensa = {m["entidad"] for m in menciones}
    paso1 = secop[secop["entidad"].isin(entidades_en_prensa)]
    paso2 = paso1[
        paso1["modalidad_de_contratacion"].str.contains("directa", case=False, na=False)
    ]
    respuestas = pd.to_numeric(paso2["respuestas_al_procedimiento"], errors="coerce")
    candidatos = paso2[respuestas.eq(0)].sort_values(
        ["precio_base", "id_del_proceso"], ascending=[False, True]
    ).reset_index(drop=True)

    with urllib.request.urlopen(f"{RAW}/Datos/noticias_contratacion_2026.json") as r:
        noticias = json.loads(r.read().decode("utf-8"))
    texto_prensa = " ".join(
        f'{n.get("titulo") or ""} {n.get("subtitulo") or ""}' for n in noticias
    ).casefold()
    referencias = candidatos["referencia_del_proceso"].fillna("").astype(str).str.strip()
    con_referencia = sum(bool(x) and len(x) >= 6 and x.casefold() in texto_prensa for x in referencias)

    assert len(menciones) == 142
    assert dict(niveles) == {"baja": 111, "media": 25, "alta": 6}
    assert len(secop) == 1000
    assert len(paso1) == 163
    assert len(candidatos) == 77
    assert con_referencia == 0
    print("Runtime recuperado: 142 entidades · 1.000 → 163 → 77 · 0/77.")
else:
    print("La bandeja sigue en memoria:", len(candidatos), "candidatos. No fue necesario reconstruirla.")

---
## 8. Tutorial visual 2 — Astra DB y CQL, paso a paso

- el **cuaderno** explica modelo y decisión;
- el **tutorial HTML** reproduce la ruta de interfaz y los nombres de los controles;
- tú ejecutas cada paso en Astra;
- vuelves con una tabla real.

**Transparencia visual.** Donde no existe una captura autenticada de una cuenta del curso usamos una **representación de interfaz claramente rotulada**, no una fotografía fingida.

Ruta del grupo: Astra DB Serverless **non-vector** + CQL Console. No instalamos Cassandra en Windows ni en Colab.

Ruta contrastada con documentación oficial vigente el **31 de agosto de 2026**. `Connection details` es el nombre vigente para bases non-vector.

In [ ]:
#@title Tutorial 2 — Astra/Cassandra: guía visual { display-mode: "form" }
tutorial(
    "https://jazaineam1.github.io/BigData2026/assets/tutoriales/astra-cassandra-paso-a-paso-v2.html",
    alto=790,
    titulo="Tutorial Astra — de cero a CQL y conexión con Python"
)

### Control antes de Python

Debes tener:

- [ ] una base Astra DB Serverless **non-vector** activa;
- [ ] keyspace `compras_claras`;
- [ ] tabla `prioridades_por_corte_departamento`;
- [ ] al menos una inserción hecha en CQL Console;
- [ ] una consulta por `corte + departamento` que funciona;
- [ ] tu **Secure Connect Bundle** descargado;
- [ ] un token de aplicación guardado fuera del cuaderno.

**Nunca pegues el token en una celda Markdown ni lo subas a GitHub.**

---
## 9. La misma tabla, ahora desde Python

El PDA pide una práctica de CRUD con Python. Primero entendiste el modelo en CQL Console; ahora una aplicación usa exactamente la misma tabla.

La siguiente celda:

1. instala el driver;
2. pide el SCB mediante el botón de carga de Colab;
3. pide el token sin mostrarlo;
4. crea la tabla si hace falta;
5. inserta la bandeja;
6. lee Bogotá;
7. actualiza el primer candidato;
8. crea y borra una fila de demostración.

Si no tienes una cuenta Astra funcional, **no inventes una ejecución**: conserva el diseño y usa el resultado de respaldo para la discusión.

### MINI FICHA DRIVER S05

| Objeto | Para qué sirve | Qué recibe | Qué deja |
|---|---|---|---|
| `Cluster(...)` | configura la conexión | SCB + autenticación | cliente del driver |
| `cluster.connect()` | abre la sesión | `Cluster` | `Session` |
| `session.prepare()` | prepara CQL parametrizado | sentencia con `?` | consulta reutilizable |
| `session.execute()` | envía CQL | sentencia + valores | filas o escritura |

**Error frecuente.** Aquí `Cluster` es un objeto del driver Python; no significa crear otro recurso en Astra.

In [ ]:
!pip install -q cassandra-driver

import importlib.metadata
print("cassandra-driver:", importlib.metadata.version("cassandra-driver"))

### DIAGNÓSTICO ASTRA S05 — si algo falla, identifica primero el síntoma

| Ves | Qué suele significar | Qué haces |
|---|---|---|
| base no está `Active` | aún está provisionando | espera/recarga; no crees otra |
| no aparece `token@cqlsh>` | CQL Console aún no conectó | espera o vuelve a abrir la consola |
| `Unauthorized` | token o permisos | genera/verifica el application token |
| SCB no conecta | bundle de otra base/región o ZIP alterado | descarga de nuevo desde **esta** base |
| consulta por `entidad` falla | **no es instalación** | revisa la clave de partición |

Antes del código recuerda cuatro objetos: **SCB = conexión**, **token = autenticación**, `"token"` = usuario literal del driver, **Cluster/Session = cliente Python**.

In [ ]:
from getpass import getpass
from datetime import date
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

try:
    from google.colab import files
    print("Sube UN solo Secure Connect Bundle (.zip) de ESTA base:")
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Debes subir exactamente un archivo SCB .zip.")
    scb = next(iter(subidos))
except ImportError:
    scb = input("Ruta al Secure Connect Bundle (.zip): ").strip()

if not scb.lower().endswith(".zip"):
    raise ValueError("El Secure Connect Bundle debe conservarse como .zip.")

token_astra = getpass("Application token de Astra (no se muestra): ").strip()
if not token_astra:
    raise ValueError("El token está vacío.")

cluster = Cluster(
    cloud={"secure_connect_bundle": scb},
    auth_provider=PlainTextAuthProvider("token", token_astra),
)
session = cluster.connect()
row = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Cassandra:", row[0] if row else "versión no disponible")

In [ ]:
cql_tabla = '''
CREATE TABLE IF NOT EXISTS compras_claras.prioridades_por_corte_departamento (
    corte date,
    departamento text,
    valor_base bigint,
    id_proceso text,
    entidad text,
    noticias_entidad int,
    nivel_menciones text,
    estado_revision text,
    url_secop text,
    criterio text,
    PRIMARY KEY ((corte, departamento), valor_base, id_proceso)
) WITH CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC);
'''
session.execute(cql_tabla)
print("Tabla lista.")

In [ ]:
def texto_seguro(valor, defecto="No definido"):
    if pd.isna(valor):
        return defecto
    texto = str(valor).strip()
    return texto if texto else defecto

insertar = session.prepare('''
INSERT INTO compras_claras.prioridades_por_corte_departamento
(corte, departamento, valor_base, id_proceso, entidad,
 noticias_entidad, nivel_menciones, estado_revision, url_secop, criterio)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''')

CORTE_CLASE = date(2026, 9, 3)

# 77 escrituras síncronas: claridad didáctica; NO es benchmark de carga masiva.
for _, f in candidatos.iterrows():
    session.execute(insertar, (
        CORTE_CLASE,
        texto_seguro(f.get("departamento_entidad")),
        int(f["precio_base"]),
        str(f["id_del_proceso"]),
        str(f["entidad"]),
        int(f["noticias_entidad"]),
        str(f["nivel_menciones"]),
        "pendiente",
        texto_seguro(f.get("urlproceso"), ""),
        "entidad en prensa; contratación directa; 0 respuestas",
    ))
print("Insertadas/actualizadas:", len(candidatos), "filas del corte", CORTE_CLASE)

In [ ]:
consulta = '''
SELECT id_proceso, entidad, valor_base, noticias_entidad, nivel_menciones, estado_revision
FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
LIMIT 5
'''

bogota = "Distrito Capital de Bogotá"
top5 = list(session.execute(consulta, (CORTE_CLASE, bogota)))
print("Top 5 —", bogota)
for f in top5:
    print(
        f.id_proceso, "|", f.entidad, "| $", f"{f.valor_base:,}",
        "| prensa:", f.noticias_entidad, f.nivel_menciones,
        "|", f.estado_revision,
    )
if not top5:
    print("No aparecieron filas. Revisa el nombre exacto del departamento.")

### EVIDENCIA INDIVIDUAL S05 — compara dos motores

Ya fijaste el top 5 esperado con pandas. Ahora pregunta lo mismo a Cassandra. La evidencia es una prueba de **corrección**, no un benchmark de velocidad.

In [ ]:
if "departamento_elegido" not in globals() or "ids_esperados_pd" not in globals():
    raise RuntimeError("Ejecuta primero CONTRATO DE RESULTADO S05.")

top5_propio = list(session.execute(consulta, (CORTE_CLASE, departamento_elegido)))
ids_cql = [str(f.id_proceso) for f in top5_propio]
coinciden_cql_pd = ids_cql == ids_esperados_pd

print("Departamento     :", departamento_elegido)
print("Esperado pandas :", ids_esperados_pd)
print("Devuelto CQL    :", ids_cql)
print("¿Coinciden?     :", "SÍ" if coinciden_cql_pd else "NO")

if not coinciden_cql_pd:
    raise AssertionError("Revisa corte, departamento, carga y orden: los dos resultados no coinciden.")

### INTERPRETACIÓN CQL S05

**Cómo se lee.** Comparamos, en orden, los IDs del top 5 calculado con pandas y el servido por Cassandra.

**Qué nos dice.** Si coinciden, la tabla query-first está sirviendo correctamente esa pregunta sobre los datos cargados.

**Qué NO permite concluir todavía.** No demuestra que Cassandra sea más rápido ni necesario para 77 filas; no hicimos una prueba de rendimiento o escala.

**Error frecuente.** Convertir una prueba de corrección en una afirmación de performance.

In [ ]:
# UPDATE: cambiamos el estado y después lo volvemos a leer.
if top5:
    objetivo = top5[0]
    session.execute('''
    UPDATE compras_claras.prioridades_por_corte_departamento
    SET estado_revision = %s
    WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
    ''', ("en_revision", CORTE_CLASE, bogota, int(objetivo.valor_base), objetivo.id_proceso))

    verificacion = session.execute('''
    SELECT estado_revision
    FROM compras_claras.prioridades_por_corte_departamento
    WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
    ''', (CORTE_CLASE, bogota, int(objetivo.valor_base), objetivo.id_proceso)).one()

    assert verificacion is not None and verificacion.estado_revision == "en_revision"
    print("UPDATE verificado:", objetivo.id_proceso, "→", verificacion.estado_revision)

In [ ]:
# DELETE sin destruir la bandeja: creamos una fila centinela y luego la borramos.
demo_insert = session.prepare('''
INSERT INTO compras_claras.prioridades_por_corte_departamento
(corte, departamento, valor_base, id_proceso, entidad, estado_revision, url_secop, criterio)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''')

session.execute(demo_insert, (
    CORTE_CLASE, "DEMO", 1, "S05-DEMO",
    "FILA DE PRACTICA", "pendiente", "", "solo para practicar DELETE"
))

antes = list(session.execute('''
SELECT id_proceso FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
''', (CORTE_CLASE, "DEMO")))

session.execute('''
DELETE FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
''', (CORTE_CLASE, "DEMO", 1, "S05-DEMO"))

despues = list(session.execute('''
SELECT id_proceso FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
''', (CORTE_CLASE, "DEMO")))

print("Antes del DELETE:", [x.id_proceso for x in antes])
print("Después del DELETE:", [x.id_proceso for x in despues])

### El error intencional

Prueba mentalmente esta consulta:

```sql
SELECT *
FROM compras_claras.prioridades_por_corte_departamento
WHERE entidad = 'MINISTERIO DEL DEPORTE';
```

La columna **sí existe**, pero la tabla no fue diseñada para localizar particiones por `entidad`.

Ese es el aprendizaje:

> **Que una columna exista no significa que este modelo esté preparado para buscar por ella.**

Si “buscar por entidad” se vuelve una consulta profesional importante, Cassandra suele pedir **otra tabla diseñada para ese patrón de acceso**, no confiar en un recorrido global.

---
<details>
<summary><strong>MÁS ADELANTE — Consistencia ajustable</strong></summary>

Cassandra replica datos. ¿Cuántas confirmaciones esperamos antes de responder? Esa decisión intercambia latencia y garantía inmediata.

No configuramos niveles hoy: primero debe quedar firme `consulta → partición → clustering`. Astra Serverless además aplica guardrails propios.
</details>

---
## 11. MongoDB Atlas y Cassandra: no compiten por el mismo papel

| Necesidad en Compras Claras | Motor de hoy | Razón |
|---|---|---|
| noticias con estructura flexible | MongoDB Atlas | documento irregular y exploración |
| clasificar y publicar una vista | MongoDB Atlas | aggregation pipeline + vista |
| cruzar con SECOP y construir la regla | pandas | integración analítica explícita |
| servir repetidamente `corte + departamento → top 5` | Cassandra/Astra | tabla modelada para ese acceso |

**PARA LLEVAR.** Cassandra no “confirma” que los 77 sean urgentes. Solo sirve de forma eficiente una priorización que ya decidiste antes.

### HOJA DE TRUCOS S05 — para consultar sin volver veinte celdas

| Necesidad | Recuerda |
|---|---|
| filtrar documentos en Atlas | `Documents` + filtro |
| encadenar transformaciones | `Aggregations` + pipeline |
| conservar la receta | `Save → Save as` |
| publicar resultado consultable | `Save → Create view` |
| localizar datos en Cassandra | primero la **partition key** |
| ordenar dentro de la partición | **clustering columns** |
| consulta de Laura | `WHERE corte = ? AND departamento = ? LIMIT 5` |
| nueva consulta por entidad | probablemente **otra tabla**, no `ALLOW FILTERING` como parche |

**PARA LLEVAR.** MongoDB favorece exploración documental flexible. Cassandra favorece patrones de acceso conocidos y repetitivos.

---
## 12. Hito de la sesión

El hito captura **tu ejecución y tu decisión**, no memoria de sintaxis.

La celda siguiente crea:

- `s05_priorizacion.csv`;
- `hito_s05_servicio_prioridades.md`.

Te preguntará dos cosas que una IA no puede sacar de la nada sin conocer tu ejecución:
qué alternativa descartaste y qué consulta importante no soporta bien tu tabla.

---
## PUENTE S05-S06 — guarda la fila que Laura abrirá después

Hasta aquí S5 respondió **qué mirar primero**. La próxima sesión ya no vuelve a construir esa decisión: toma uno de tus procesos priorizados y pregunta **qué relaciones existen alrededor de él**.

El archivo `s05_ancla_s06.json` conserva el proceso elegido, su entidad y el contexto heredado de las noticias. Si lo conservas, S6 comienza exactamente desde tu propia ejecución. Si lo pierdes, S6 podrá reconstruir la misma bandeja sin bloquearte.

In [ ]:
import json

if "top5_esperado_pd" in globals() and len(top5_esperado_pd):
    fila_ancla = top5_esperado_pd.iloc[0]
else:
    fila_ancla = candidatos.iloc[0]

ancla_s06 = {
    "id_proceso": str(fila_ancla["id_del_proceso"]),
    "referencia": str(fila_ancla.get("referencia_del_proceso", "")),
    "entidad": str(fila_ancla["entidad"]),
    "nit_entidad": str(fila_ancla.get("nit_entidad", "")),
    "departamento": str(fila_ancla.get("departamento_entidad", "")),
    "valor_base": int(float(fila_ancla["precio_base"])),
    "modalidad": str(fila_ancla.get("modalidad_de_contratacion", "")),
    "noticias_entidad": int(fila_ancla["noticias_entidad"]),
    "nivel_menciones": str(fila_ancla["nivel_menciones"]),
    "url_secop": str(fila_ancla.get("urlproceso", "")),
    "origen": "bandeja operacional S05: 1.000→163→77",
}

with open("s05_ancla_s06.json", "w", encoding="utf-8") as f:
    json.dump(ancla_s06, f, ensure_ascii=False, indent=2)

print("Ancla S6 lista:")
print(json.dumps(ancla_s06, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download("s05_ancla_s06.json")
except Exception:
    print("Archivo guardado como s05_ancla_s06.json")

In [ ]:
alternativa = input("Alternativa de diseño descartada: ").strip()
razon_alternativa = input("¿Por qué la descartaste para la consulta de Laura?: ").strip()
consulta_no_soportada = input("Una consulta profesional que esta tabla NO soporta bien: ").strip()
nueva_particion = input("Si fuera frecuente, ¿qué dato(s) usarías para localizar la nueva partición?: ").strip()

departamento_hito = globals().get("departamento_elegido", "No seleccionado")
ids_pd_hito = globals().get("ids_esperados_pd", [])
ids_cql_hito = globals().get("ids_cql", [])
coincidencia_hito = globals().get("coinciden_cql_pd", False)
primer_id = str(candidatos.iloc[0]["id_del_proceso"])
primer_entidad = str(candidatos.iloc[0]["entidad"])
primer_valor = int(candidatos.iloc[0]["precio_base"])
primer_noticias = int(candidatos.iloc[0]["noticias_entidad"])
primer_nivel = str(candidatos.iloc[0]["nivel_menciones"])

candidatos.to_csv("s05_priorizacion.csv", index=False, encoding="utf-8")

hito = f'''# Hito S05 — De la priorización al servicio

## Resultado propio de Atlas
- Fuente: {"vista real de Atlas" if vista_real else "respaldo; falta evidencia de vista real"}
- Alta / media / baja: {niveles.get("alta", 0)} / {niveles.get("media", 0)} / {niveles.get("baja", 0)}

## Regla de priorización
- Procesos iniciales: {len(secop)}
- Coincidencias por entidad: {len(paso1)}
- Candidatos: {len(candidatos)}
- Primer candidato: {primer_id} — {primer_entidad} — $ {primer_valor:,}
- Contexto desde Atlas: {primer_noticias} noticias — nivel {primer_nivel}

## Límite
Referencias de proceso citadas literalmente en prensa: {con_referencia} de {len(candidatos)}.
La evidencia periodística usada es por entidad; no demuestra irregularidad del contrato específico.

## Query-first
PRIMARY KEY ((corte, departamento), valor_base, id_proceso)
CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC)

## Evidencia individual de corrección
- Departamento: {departamento_hito}
- Top esperado con pandas: {ids_pd_hito}
- Top devuelto por Cassandra: {ids_cql_hito if ids_cql_hito else "NO VERIFICADO EN ASTRA"}
- Coincidencia exacta de IDs y orden: {"sí" if coincidencia_hito else "no verificada"}

## Alternativa descartada
{alternativa or "PENDIENTE"}

Razón: {razon_alternativa or "PENDIENTE"}

## Consulta no soportada
{consulta_no_soportada or "PENDIENTE"}

Nueva localización de partición si fuera frecuente: {nueva_particion or "PENDIENTE"}

## Decisión
MongoDB transforma documentos; pandas materializa una regla auditable; Cassandra sirve una proyección para una consulta repetitiva conocida.
'''

with open("hito_s05_servicio_prioridades.md", "w", encoding="utf-8") as f:
    f.write(hito)
print(hito)
print("
Archivos creados: s05_priorizacion.csv, hito_s05_servicio_prioridades.md")

In [ ]:
try:
    from google.colab import files
    files.download("hito_s05_servicio_prioridades.md")
    files.download("s05_priorizacion.csv")
except ImportError:
    print("Fuera de Colab: descarga los archivos desde el explorador de tu entorno.")

In [ ]:
#@title Cerrar conexiones { display-mode: "form" }
# Buena práctica: cerrar clientes al terminar la sesión.
try:
    cluster.shutdown()
    print("Conexión Cassandra cerrada.")
except Exception:
    pass

try:
    if "client" in globals():
        client.close()
        print("Conexión MongoDB cerrada.")
except Exception:
    pass

In [ ]:
#@title Cerrar conexiones { display-mode: "form" }
# Buena práctica: cerrar clientes al terminar la sesión.
try:
    cluster.shutdown()
    print("Conexión Cassandra cerrada.")
except Exception:
    pass

try:
    if "client" in globals():
        client.close()
        print("Conexión MongoDB cerrada.")
except Exception:
    pass

In [ ]:
#@title Cerrar conexiones { display-mode: "form" }
# Buena práctica: cerrar clientes al terminar la sesión.
try:
    cluster.shutdown()
    print("Conexión Cassandra cerrada.")
except Exception:
    pass

try:
    if "client" in globals():
        client.close()
        print("Conexión MongoDB cerrada.")
except Exception:
    pass

In [ ]:
#@title Cerrar conexiones { display-mode: "form" }
# Buena práctica: cerrar clientes al terminar la sesión.
try:
    cluster.shutdown()
    print("Conexión Cassandra cerrada.")
except Exception:
    pass

try:
    if "client" in globals():
        client.close()
        print("Conexión MongoDB cerrada.")
except Exception:
    pass

## Rúbrica de calidad del hito

| Criterio | Completo | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Vista Atlas | vista real + 142 / 6-25-111 | respaldo declarado | números sin fuente | 15 |
| Regla + límite | 1.000→163→77 y explica 0/77 | números sin límite concreto | llama “irregulares” a los 77 | 20 |
| Query-first | PK/clustering explicados desde la consulta | copia diseño sin justificar | clave no sirve esa consulta | 20 |
| Evidencia individual | departamento + top pandas + top CQL + coincidencia | top pandas y contingencia Astra declarada | no hay resultado propio | 20 |
| Alternativa | alternativa + razón de descarte | alternativa sin razón | no hay alternativa | 15 |
| Consulta no soportada | consulta distinta + nueva localización | consulta sin rediseño | afirma que cualquier filtro funciona | 10 |

Las autoevaluaciones son **formativas**. La evidencia revisable es el hito producido por la ejecución.

## Lo que sigue

S5 terminó con una fila que Laura puede justificar y consultar repetidamente. Pero una fila sigue siendo una fila.

La próxima sesión empieza cuando Laura abre **ese proceso** y pregunta:

> **“Ya sé por qué este proceso llegó primero a mi bandeja. Antes de asignarlo a un auditor, ¿qué relaciones alrededor de su entidad y sus procesos históricos necesito ver?”**

El candidato de S5 será el **ancla**. Los procesos históricos adjudicados aportarán el contexto que ese candidato todavía no tiene: proveedores, otras contrataciones y conexiones con otras entidades.

```text
S3  evidencia documental
 ↓
S4  persistencia compartida
 ↓
S5  qué revisar primero
 ↓
S6  qué hay alrededor de lo que voy a revisar
```

Ahí aparece Neo4j. No para declarar irregularidades, sino para hacer de las **relaciones** una parte explícita de la revisión.